In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv("/content/ArtificialPatientTreatment.csv")

In [5]:
df.head(20)

,patient,action,org:resource,DateTime
0,patient 0,First consult,Dr. Anna,2017-01-02 11:40:11
1,patient 0,Blood test,Lab,2017-01-02 12:47:33
2,patient 0,Physical test,Nurse Jesse,2017-01-02 12:53:50
3,patient 0,Second consult,Dr. Anna,2017-01-02 16:21:06
4,patient 0,Surgery,Dr. Charlie,2017-01-05 13:23:09
5,patient 0,Final consult,Dr. Ben,2017-01-09 08:29:28
6,patient 1,First consult,Dr. Anna,2017-01-02 12:50:35
7,patient 1,Physical test,Nurse Jesse,2017-01-02 13:59:14
8,patient 1,Blood test,Lab,2017-01-02 14:20:19
9,patient 1,X-ray scan,Team 1,2017-01-06 09:13:40


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   patient        690 non-null    object
 1    action        690 non-null    object
 2    org:resource  690 non-null    object
 3    DateTime      690 non-null    object
dtypes: object(4)
memory usage: 21.7+ KB


In [7]:
df.describe()

,patient,action,org:resource,DateTime
count,690,690,690,690
unique,100,8,13,690
top,patient 1,First consult,Dr. Anna,2017-01-02 11:40:11
freq,7,100,158,1


In [8]:
df.shape

(690, 4)

In [9]:
df.isnull().sum()

,0
patient,0
action,0
org:resource,0
DateTime,0


In [10]:
df.columns = df.columns.str.strip()

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690 entries, 0 to 689
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   patient       690 non-null    object
 1   action        690 non-null    object
 2   org:resource  690 non-null    object
 3   DateTime      690 non-null    object
dtypes: object(4)
memory usage: 21.7+ KB


In [12]:
df.head()

,patient,action,org:resource,DateTime
0,patient 0,First consult,Dr. Anna,2017-01-02 11:40:11
1,patient 0,Blood test,Lab,2017-01-02 12:47:33
2,patient 0,Physical test,Nurse Jesse,2017-01-02 12:53:50
3,patient 0,Second consult,Dr. Anna,2017-01-02 16:21:06
4,patient 0,Surgery,Dr. Charlie,2017-01-05 13:23:09


In [13]:
df['DateTimeClean'] = pd.to_datetime(df['DateTime'])

In [14]:
df.head()

,patient,action,org:resource,DateTime,DateTimeClean
0,patient 0,First consult,Dr. Anna,2017-01-02 11:40:11,2017-01-02 11:40:11
1,patient 0,Blood test,Lab,2017-01-02 12:47:33,2017-01-02 12:47:33
2,patient 0,Physical test,Nurse Jesse,2017-01-02 12:53:50,2017-01-02 12:53:50
3,patient 0,Second consult,Dr. Anna,2017-01-02 16:21:06,2017-01-02 16:21:06
4,patient 0,Surgery,Dr. Charlie,2017-01-05 13:23:09,2017-01-05 13:23:09


In [15]:
df = df.sort_values(by=['patient','DateTimeClean'])

In [16]:
df.head()

,patient,action,org:resource,DateTime,DateTimeClean
0,patient 0,First consult,Dr. Anna,2017-01-02 11:40:11,2017-01-02 11:40:11
1,patient 0,Blood test,Lab,2017-01-02 12:47:33,2017-01-02 12:47:33
2,patient 0,Physical test,Nurse Jesse,2017-01-02 12:53:50,2017-01-02 12:53:50
3,patient 0,Second consult,Dr. Anna,2017-01-02 16:21:06,2017-01-02 16:21:06
4,patient 0,Surgery,Dr. Charlie,2017-01-05 13:23:09,2017-01-05 13:23:09


In [17]:
df['NextActivity'] = df.groupby('patient')['action'].shift(-1)

In [18]:
df['NextActivity'] = df['NextActivity'].fillna('[END]')

In [19]:
df['NextActivity'].head(20)

,NextActivity
0,Blood test
1,Physical test
2,Second consult
3,Surgery
4,Final consult
5,[END]
6,Physical test
7,Blood test
8,X-ray scan
9,Second consult


In [29]:
df['Next_Timestamp'] = df.groupby('patient')['DateTimeClean'].shift(-1)

In [30]:
df['Next_Timestamp'] = df['Next_Timestamp'].fillna('[END]')

In [34]:
next_timestamp_raw = df.groupby('patient')['DateTimeClean'].shift(-1)
df['Duration_Minutes'] = (next_timestamp_raw - df['DateTimeClean']).dt.total_seconds() / 60

In [36]:
df['Transition'] = df['action']+"->"+df['NextActivity']

In [37]:
df['Transition'].head(20)

,Transition
0,First consult-> Blood test
1,Blood test-> Physical test
2,Physical test-> Second consult
3,Second consult-> Surgery
4,Surgery-> Final consult
5,Final consult->[END]
6,First consult-> Physical test
7,Physical test-> Blood test
8,Blood test-> X-ray scan
9,X-ray scan-> Second consult


In [38]:
bottlenecks = df.groupby('Transition')['Duration_Minutes'].agg(['mean','median','count']).reset_index()
bottlenecks = bottlenecks.sort_values(by='mean',ascending=False)

In [39]:
print("\nBottlenecks Average transition Time:")
print(bottlenecks)


Bottlenecks Average transition Time:
                         Transition         mean       median  count
12         Second consult-> Surgery  5654.834167  5897.683333     20
6        First consult-> X-ray scan  5240.754321  5717.183333     27
16      X-ray scan-> Second consult  4771.534946  5656.133333     31
2           Blood test-> X-ray scan  4342.154688  4368.625000     32
9    Physical test-> Second consult  3987.615741  4250.350000     36
1       Blood test-> Second consult  3769.540909  3319.850000     33
10       Physical test-> X-ray scan  3182.035484  2820.783333     31
7          Medicine-> Final consult  3080.343958  2720.783333     80
13          Surgery-> Final consult  2882.051667  2954.400000     20
5     First consult-> Physical test  2203.796465  1463.150000     33
0        Blood test-> Physical test   761.521429   958.933333     35
11        Second consult-> Medicine   545.180625   144.533333     80
15       X-ray scan-> Physical test   378.915104    43.916667    